# M3 – Manual labeling (Colab)

Notebook gồm 3 phần: **MAKE → MANUAL MATCHING → CHECK & COLLECT**.

**Quan trọng**
- Notebook dùng đúng đường dẫn `src/labeling/label_sheet.py` và `src/labeling/manual_match.py`.
- Không tự `git pull` khi repo đã tồn tại, để tránh ghi đè `manual_match.py` bạn vừa thay.
- MAKE không ghi đè các file đã có trong thư mục `*_filled`.
- Khi manual matching, nhập `q` để kết thúc một paper và lưu CSV.
- Chạy lần lượt từ trên xuống. Nếu cell kiểm tra báo lỗi, dừng tại đó và gửi nguyên output lỗi.


## 0. Cấu hình

In [ ]:
# ===== CONFIG =====
MSSV = "24127465"
BRANCH = "m3-manual-labeling"
PAPERS = [
    "2602.11395",  # test - manual
    "2602.12886",  # validation - manual
    "2602.11055",  # train - manual
    "2602.10677",  # train - manual
    "2602.11244",  # train - manual
]

REPO_URL = "https://github.com/VincesPowder/hcmus-project-arxiv.git"
REPO_DIR = "/content/hcmus-project-arxiv"

BIBTEX_DIR = (
    "/content/drive/MyDrive/arXiv-metadata-hcmus/"
    "milestone2_outputs/m2_full_v5/exports/20261120-203957-9b2330/tables/bibtex"
)
REFERENCES_PARQUET = (
    "/content/drive/MyDrive/arXiv-metadata-hcmus/"
    "2602.09870-2602.14869/work/tables/references.parquet"
)
SELECTION_CSV = (
    "/content/drive/MyDrive/arXiv-metadata-hcmus/"
    "m3_selection/selection.csv"
)

OUTPUT_DIR = (
    f"/content/drive/MyDrive/arXiv-metadata-hcmus/labeling/{MSSV}"
)
FILLED_DIR = (
    f"/content/drive/MyDrive/arXiv-metadata-hcmus/labeling/{MSSV}_filled"
)
LABELS_OUT = f"{REPO_DIR}/labels/labels_manual_{MSSV}.csv"

# Không dùng MAKE_FRESH để không xóa/ghi đè dữ liệu đã làm.
MAKE_FRESH = False

print("Configuration loaded.")
print("MSSV:", MSSV)
print("Branch:", BRANCH)
print("Papers:", PAPERS)


Configuration loaded.
MSSV: 24127465
Branch: m3-manual-labeling
Papers: ['2602.11395', '2602.12886', '2602.11055', '2602.10677', '2602.11244']


## 1. Kết nối Drive, chuẩn bị repo và kiểm tra môi trường

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import shutil
import runpy

repo_path = Path(REPO_DIR)

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
        check=True
    )
else:
    print("Repo đã tồn tại; giữ nguyên working tree, KHÔNG pull/reset để không ghi đè file manual_match.py.")
    current_branch = subprocess.check_output(
        ["git", "-C", REPO_DIR, "branch", "--show-current"],
        text=True
    ).strip()
    if current_branch != BRANCH:
        raise RuntimeError(
            f"Repo đang ở branch {current_branch!r}, cần {BRANCH!r}. "
            "Không tự checkout để tránh làm mất thay đổi local."
        )

os.chdir(REPO_DIR)

LABEL_SCRIPT = "src/labeling/label_sheet.py"
MATCH_SCRIPT = "src/labeling/manual_match.py"
SRC_DIR = str(Path(REPO_DIR) / "src")

# Script con cần nhìn thấy src/text_clean.py.
os.environ["PYTHONPATH"] = SRC_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Cài dependencies tối thiểu cho notebook/data.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "pyarrow"],
    check=True
)

for required_file in [LABEL_SCRIPT, MATCH_SCRIPT, "src/text_clean.py"]:
    if not Path(required_file).is_file():
        raise FileNotFoundError(
            f"Thiếu {required_file}. Repo/branch hiện tại chưa có đủ file."
        )
    print("FOUND:", required_file)

for required_path in [BIBTEX_DIR, REFERENCES_PARQUET, SELECTION_CSV]:
    if not Path(required_path).exists():
        raise FileNotFoundError(f"Không tìm thấy dữ liệu đầu vào: {required_path}")
    print("FOUND:", required_path)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(FILLED_DIR).mkdir(parents=True, exist_ok=True)
Path(REPO_DIR, "labels").mkdir(parents=True, exist_ok=True)

# Compile các file; lỗi cú pháp sẽ được báo ngay tại đây.
subprocess.run(
    [sys.executable, "-m", "py_compile", LABEL_SCRIPT, MATCH_SCRIPT, "src/text_clean.py"],
    check=True
)

print("\nENVIRONMENT CHECK PASSED")
print("Repo:", REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


Mounted at /content/drive
FOUND: src/labeling/label_sheet.py
FOUND: src/labeling/manual_match.py
FOUND: src/text_clean.py
FOUND: /content/drive/MyDrive/arXiv-metadata-hcmus/milestone2_outputs/m2_full_v5/exports/20261120-203957-9b2330/tables/bibtex
FOUND: /content/drive/MyDrive/arXiv-metadata-hcmus/2602.09870-2602.14869/work/tables/references.parquet
FOUND: /content/drive/MyDrive/arXiv-metadata-hcmus/m3_selection/selection.csv

ENVIRONMENT CHECK PASSED
Repo: /content/hcmus-project-arxiv
Branch: m3-manual-labeling
Commit: f21b7c6


## 1.1. Kiểm tra CLI của `label_sheet.py`

Cell này chỉ đọc trợ giúp CLI, không tạo hay sửa CSV. Nếu lệnh help báo lỗi, dừng và gửi output.

In [ ]:
help_result = subprocess.run(
    [sys.executable, LABEL_SCRIPT, "--help"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print(help_result.stdout)
if help_result.returncode != 0:
    raise RuntimeError("Không đọc được CLI help của label_sheet.py; chưa chạy MAKE.")


usage: label_sheet.py [-h] {make,collect} ...

100% manual labelling with spreadsheets

positional arguments:
  {make,collect}
    make          create manual labelling sheets
    collect       validate filled sheets and create labels.csv

options:
  -h, --help      show this help message and exit



## 2. MAKE – tạo citation sheets và references sheets

Chỉ chạy cell này sau khi bước kiểm tra môi trường thành công. Output được in trực tiếp để nếu lỗi sẽ thấy traceback gốc.

In [ ]:
make_cmd = [
    sys.executable, LABEL_SCRIPT, "make",
    "--mssv", MSSV,
    "--bibtex", BIBTEX_DIR,
    "--references", REFERENCES_PARQUET,
    "--selection", SELECTION_CSV,
    "--out-dir", OUTPUT_DIR,
    "--papers", *PAPERS,
]

print("MAKE command:")
print(" ".join(make_cmd))
make_result = subprocess.run(
    make_cmd,
    cwd=REPO_DIR,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print(make_result.stdout)

if make_result.returncode != 0:
    raise RuntimeError(
        f"MAKE thất bại (exit={make_result.returncode}). "
        "Đọc traceback ngay phía trên; không chạy các bước sau."
    )

print("MAKE hoàn tất.")


MAKE command:
/usr/bin/python3 src/labeling/label_sheet.py make --mssv 24127465 --bibtex /content/drive/MyDrive/arXiv-metadata-hcmus/milestone2_outputs/m2_full_v5/exports/20261120-203957-9b2330/tables/bibtex --references /content/drive/MyDrive/arXiv-metadata-hcmus/2602.09870-2602.14869/work/tables/references.parquet --selection /content/drive/MyDrive/arXiv-metadata-hcmus/m3_selection/selection.csv --out-dir /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465 --papers 2602.11395 2602.12886 2602.11055 2602.10677 2602.11244
NOTE: 'Best suggestion' lines below are hints only. They are also written to the `best_suggestion` CSV column, but they are NOT ground truth: check every entry by hand against <paper>_references.csv.

2602.11395: 27 entries, 37 references, ~20 entries with a strong suggestion (a rough guide to how many matches to expect)
  ho2020denoising: Best suggestion: 2006.11239
  song2021scorebased: Best suggestion: 2011.13456
  dhariwal2021diffusion: Best suggestion: 2

## 2.1. Kiểm tra CSV do MAKE tạo

In [ ]:
for paper in PAPERS:
    citation_file = Path(OUTPUT_DIR) / f"{paper}.csv"
    refs_file = Path(OUTPUT_DIR) / f"{paper}_references.csv"

    if not citation_file.is_file():
        raise FileNotFoundError(f"MAKE chưa tạo citation CSV: {citation_file}")
    if not refs_file.is_file():
        raise FileNotFoundError(f"MAKE chưa tạo references CSV: {refs_file}")

    citations = pd.read_csv(citation_file, dtype=str, keep_default_na=False)
    refs = pd.read_csv(refs_file, dtype=str, keep_default_na=False)

    print(f"\n{paper}")
    print("  citations:", len(citations), "| columns:", citations.columns.tolist())
    print("  references:", len(refs), "| columns:", refs.columns.tolist())

    required_citation_cols = {
        "citation_key", "entry_title", "entry_authors",
        "entry_year", "best_suggestion", "ref_arxiv_id",
        "match_detail", "note"
    }
    missing = required_citation_cols - set(citations.columns)
    if missing:
        raise ValueError(f"{paper}: citation CSV thiếu cột {sorted(missing)}")

    required_ref_cols = {
        "ref_arxiv_id", "ref_title", "ref_authors", "ref_submission_date"
    }
    missing_ref = required_ref_cols - set(refs.columns)
    if missing_ref:
        raise ValueError(f"{paper}: references CSV thiếu cột {sorted(missing_ref)}")

print("\nMAKE CSV validation passed.")



2602.11395
  citations: 27 | columns: ['stt', 'arxiv_id', 'citation_key', 'entry_title', 'entry_authors', 'entry_year', 'entry_venue', 'best_suggestion', 'ref_arxiv_id', 'match_detail', 'note']
  references: 37 | columns: ['ref_arxiv_id', 'ref_title', 'ref_authors', 'ref_submission_date']

2602.12886
  citations: 37 | columns: ['stt', 'arxiv_id', 'citation_key', 'entry_title', 'entry_authors', 'entry_year', 'entry_venue', 'best_suggestion', 'ref_arxiv_id', 'match_detail', 'note']
  references: 25 | columns: ['ref_arxiv_id', 'ref_title', 'ref_authors', 'ref_submission_date']

2602.11055
  citations: 63 | columns: ['stt', 'arxiv_id', 'citation_key', 'entry_title', 'entry_authors', 'entry_year', 'entry_venue', 'best_suggestion', 'ref_arxiv_id', 'match_detail', 'note']
  references: 23 | columns: ['ref_arxiv_id', 'ref_title', 'ref_authors', 'ref_submission_date']

2602.10677
  citations: 37 | columns: ['stt', 'arxiv_id', 'citation_key', 'entry_title', 'entry_authors', 'entry_year', 'entry

## 2.2. Chuẩn bị thư mục filled

Sao chép citation CSV mới sang thư mục filled **chỉ khi file filled chưa tồn tại**. Nếu bạn đã bắt đầu gán nhãn, file cũ được giữ nguyên.

In [ ]:
for paper in PAPERS:
    source = Path(OUTPUT_DIR) / f"{paper}.csv"
    target = Path(FILLED_DIR) / f"{paper}.csv"

    if target.exists():
        print("KEEP existing filled file:", target)
    else:
        shutil.copy2(source, target)
        print("CREATED filled file:", target)

print("\nFilled directory:", FILLED_DIR)


CREATED filled file: /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled/2602.11395.csv
CREATED filled file: /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled/2602.12886.csv
CREATED filled file: /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled/2602.11055.csv
CREATED filled file: /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled/2602.10677.csv
CREATED filled file: /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled/2602.11244.csv

Filled directory: /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled


## 3. MANUAL MATCHING

Chạy cell này để gán nhãn lần lượt 5 paper. Mỗi paper dùng:
- citation CSV trong `FILLED_DIR`;
- references CSV trong `OUTPUT_DIR`.

Trong từng paper:
- `1`–`5`: chọn candidate;
- `m`: nhập ID thủ công;
- `n`: xác nhận không có reference;
- `s`: bỏ qua citation;
- `q`: kết thúc paper hiện tại và lưu CSV.

**Lưu ý:** Nếu bạn dừng hẳn phiên Colab giữa lúc đang gán một paper, hãy nhập `q` trước để lưu các thay đổi của phiên đó.

In [ ]:
\yyy
yyy
yyyyyyy
yyyimport sys
import runpy
from pathlib import Path

for paper in PAPERS:
    citation_csv = str(Path(FILLED_DIR) / f"{paper}.csv")
    references_csv = str(Path(OUTPUT_DIR) / f"{paper}_references.csv")

    if not Path(citation_csv).is_file():
        raise FileNotFoundError(citation_csv)
    if not Path(references_csv).is_file():
        raise FileNotFoundError(references_csv)

    print("\n" + "#" * 100)
    print("BẮT ĐẦU MANUAL MATCHING:", paper)
    print("#" * 100)

    old_argv = sys.argv[:]
    try:
        sys.argv = [
            MATCH_SCRIPT,
            "--csv", citation_csv,
            "--references", references_csv,
        ]
        runpy.run_path(MATCH_SCRIPT, run_name="__main__")
    finally:
        sys.argv = old_argv

    print("\nĐã kết thúc phiên matching:", paper)
    print("CSV hiện tại:", citation_csv)

print("\nĐã chạy hết danh sách paper.")


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
    Title   : Anomalous elasticity and plastic screening in amorphous solids.
    Authors : A. Lemaître, C. Mondal, Michael Moshe, I. Procaccia, Saikat Roy, Keren Screiber-Re'em
    Year    : 2021

[4] SCORE = 37
    arXiv ID: 2111.11198
    Title   : Direct measurement of dipoles in anomalous elasticity of amorphous solids.
    Authors : Bhanu Prasad Bhowmik, Michael Moshe, I. Procaccia
    Year    : 2021

[5] SCORE = 33
    arXiv ID: 2310.09942
    Title   : Odd dipole screening in disordered matter
    Authors : Y. Cohen, Amit Schiller, Dong Wang, Joshua Dijksman, M. Moshe
    Year    : 2023

LỰA CHỌN:
  1-5 = chọn candidate trong TOP 5
  f   = tìm trong TOÀN BỘ references theo ID / title / author / year
  a   = hiển thị TOÀN BỘ references
  m   = nhập arXiv ID thủ công nếu đã biết
  n   = NONE, chỉ chọn sau khi đã kiểm tra toàn bộ references
  s   = SKIP, chưa chắc / làm sau
  q   = QUIT

Chọn: 1

Bạn chọn: 2209.05311
Xác nhậ

## 3.1. Kiểm tra tiến độ gán nhãn

In [ ]:
progress = []

for paper in PAPERS:
    path = Path(FILLED_DIR) / f"{paper}.csv"
    if not path.exists():
        progress.append((paper, 0, 0, "MISSING"))
        continue

    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "ref_arxiv_id" not in df.columns:
        progress.append((paper, len(df), 0, "MISSING ref_arxiv_id"))
        continue

    values = df["ref_arxiv_id"].astype(str).str.strip().str.lower()
    labeled = ((values != "") & (values != "nan")).sum()
    progress.append((paper, len(df), int(labeled), "OK"))

progress_df = pd.DataFrame(
    progress,
    columns=["arxiv_id", "total_citations", "labeled_count", "status"]
)
display(progress_df)

print("Total labeled:", int(progress_df["labeled_count"].sum()))
print("Target: at least 20 labeled per paper and 100 total.")


,arxiv_id,total_citations,labeled_count,status
0,2602.11395,27,27,OK
1,2602.12886,37,37,OK
2,2602.11055,63,63,OK
3,2602.10677,37,37,OK
4,2602.11244,55,55,OK


Total labeled: 219
Target: at least 20 labeled per paper and 100 total.


## 4. CHECK & COLLECT

Cell kiểm tra yêu cầu tối thiểu trước khi collect: mỗi paper có ít nhất 20 nhãn (khác rỗng). `none` được tính là nhãn vì đó là quyết định ground truth hợp lệ.

In [ ]:
problems = []
total_labeled = 0

for paper in PAPERS:
    path = Path(FILLED_DIR) / f"{paper}.csv"
    if not path.exists():
        problems.append(f"{paper}: thiếu {path.name}")
        continue

    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "ref_arxiv_id" not in df.columns:
        problems.append(f"{paper}: thiếu cột ref_arxiv_id")
        continue

    values = df["ref_arxiv_id"].astype(str).str.strip().str.lower()
    labeled = int(((values != "") & (values != "nan")).sum())
    total_labeled += labeled

    print(f"{paper}: {labeled}/{len(df)} labeled")

    if labeled < 20:
        problems.append(f"{paper}: mới có {labeled} nhãn, cần ít nhất 20")

print("TOTAL:", total_labeled)

if problems:
    print("\nCHƯA ĐỦ ĐIỀU KIỆN:")
    for problem in problems:
        print("-", problem)
    raise RuntimeError("Chưa collect vì chưa đạt yêu cầu labeling.")

if total_labeled < 100:
    raise RuntimeError(f"Tổng nhãn {total_labeled}, cần ít nhất 100.")

print("\nĐã đạt ngưỡng số lượng nhãn.")


2602.11395: 27/27 labeled
2602.12886: 37/37 labeled
2602.11055: 63/63 labeled
2602.10677: 37/37 labeled
2602.11244: 55/55 labeled
TOTAL: 219

Đã đạt ngưỡng số lượng nhãn.


## 4.1. Collect nhãn thành file tổng

In [24]:
collect_cmd = [
    sys.executable, LABEL_SCRIPT, "collect",
    "--mssv", MSSV,
    "--sheets-dir", FILLED_DIR,
    "--references", REFERENCES_PARQUET,
    "--selection", SELECTION_CSV,
    "--out", LABELS_OUT,
]

print("COLLECT command:")
print(" ".join(collect_cmd))

collect_result = subprocess.run(
    collect_cmd,
    cwd=REPO_DIR,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print(collect_result.stdout)

if collect_result.returncode != 0:
    raise RuntimeError(
        f"COLLECT thất bại (exit={collect_result.returncode}). "
        "Đọc traceback ngay phía trên."
    )

print("COLLECT hoàn tất:", LABELS_OUT)


COLLECT command:
/usr/bin/python3 src/labeling/label_sheet.py collect --mssv 24127465 --sheets-dir /content/drive/MyDrive/arXiv-metadata-hcmus/labeling/24127465_filled --references /content/drive/MyDrive/arXiv-metadata-hcmus/2602.09870-2602.14869/work/tables/references.parquet --selection /content/drive/MyDrive/arXiv-metadata-hcmus/m3_selection/selection.csv --out /content/hcmus-project-arxiv/labels/labels_manual_24127465.csv
wrote 219 labels -> /content/hcmus-project-arxiv/labels/labels_manual_24127465.csv

COLLECT hoàn tất: /content/hcmus-project-arxiv/labels/labels_manual_24127465.csv


## 4.2. Kiểm tra file labels cuối cùng

In [25]:
labels_path = Path(LABELS_OUT)

if not labels_path.exists():
    raise FileNotFoundError(f"Không thấy file output: {labels_path}")

labels = pd.read_csv(
    labels_path,
    dtype=str,
    keep_default_na=False
)

print("Output:", labels_path)
print("Rows:", len(labels))
print("Columns:", labels.columns.tolist())
display(labels.head(10))

# Schema đúng của MANUAL LABELS
expected = [
    "arxiv_id",
    "citation_key",
    "ref_arxiv_id",
    "source",
    "annotator"
]

missing = [c for c in expected if c not in labels.columns]

if missing:
    raise ValueError(
        f"File manual labels thiếu các cột: {missing}"
    )

# Kiểm tra tối thiểu của M3
counts = labels.groupby("arxiv_id").size()

if len(counts) < 5:
    raise ValueError(
        f"Chỉ có {len(counts)} publication, cần ít nhất 5."
    )

if (counts < 20).any():
    raise ValueError(
        f"Có publication dưới 20 citation:\n{counts[counts < 20]}"
    )

if len(labels) < 100:
    raise ValueError(
        f"Tổng chỉ có {len(labels)} labels, cần ít nhất 100."
    )

print("\nManual labels schema check passed.")
print(f"Total labels: {len(labels)}")

print("\nLabels per publication:")
print(counts)

Output: /content/hcmus-project-arxiv/labels/labels_manual_24127465.csv
Rows: 219
Columns: ['arxiv_id', 'citation_key', 'ref_arxiv_id', 'source', 'annotator']


,arxiv_id,citation_key,ref_arxiv_id,source,annotator
0,2602.11395,ho2020denoising,2006.11239,manual,24127465
1,2602.11395,song2021scorebased,2011.13456,manual,24127465
2,2602.11395,dhariwal2021diffusion,2105.05233,manual,24127465
3,2602.11395,ho2022classifierfree,2207.12598,manual,24127465
4,2602.11395,ye2024tfg,2409.15761,manual,24127465
5,2602.11395,radhakrishnan2024mechanism,2502.03708,manual,24127465
6,2602.11395,karras2022elucidating,,manual,24127465
7,2602.11395,kwon2023diffusion,2210.10960,manual,24127465
8,2602.11395,song2021ddim,2010.02502,manual,24127465
9,2602.11395,jeong2024hspace,2303.15403,manual,24127465



Manual labels schema check passed.
Total labels: 219

Labels per publication:
arxiv_id
2602.10677    37
2602.11055    63
2602.11244    55
2602.11395    27
2602.12886    37
dtype: int64


In [26]:
import pandas as pd
d = pd.read_csv("/content/hcmus-project-arxiv/labels/labels_manual_24127465.csv", dtype=str, keep_default_na=False)
m = d[d.ref_arxiv_id != ""]
print(m.groupby("arxiv_id").size(), len(m))

arxiv_id
2602.10677    22
2602.11055    23
2602.11244    48
2602.11395    21
2602.12886    24
dtype: int64 138


## Ghi chú
- Notebook không thể đảm bảo không bao giờ có lỗi do dữ liệu Drive, quyền truy cập, hoặc khác biệt CLI trong phiên bản `label_sheet.py`; thay vào đó, các cell kiểm tra đầu vào và schema sẽ dừng sớm với thông báo cụ thể.
- Repo đã tồn tại thì notebook không tự pull/reset để bảo toàn `manual_match.py` bạn vừa thay.
- Nếu mở notebook trên runtime Colab hoàn toàn mới, repo sẽ được clone từ GitHub. Khi đó cần đưa bản `manual_match.py` đã sửa vào `src/labeling/manual_match.py` trước khi chạy matching nếu bản sửa chưa được push lên branch.
